# EDA — Olist Brazilian E-Commerce (Mini Lakehouse)

Phân tích khám phá bộ dữ liệu Olist (9 bảng) phục vụ chương **Phân tích & xử lý
dữ liệu**. Notebook đọc trực tiếp `dataset/*.csv` (read-only, tách biệt hoàn toàn
khỏi stack Docker) và xuất các biểu đồ ra `figures/` để chèn báo cáo.

**Nội dung:** Tổng quan → Profiling null → Phân phối → Insight nghiệp vụ →
Vấn đề chất lượng dữ liệu (DQ) → Bằng chứng pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["axes.grid"] = True

# notebook chạy từ notebooks/ -> dataset ở ../dataset; fallback nếu chạy ở repo root
DATASET_DIR = "../dataset" if os.path.isdir("../dataset") else "dataset"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

orders    = pd.read_csv(f"{DATASET_DIR}/olist_orders_dataset.csv")
items     = pd.read_csv(f"{DATASET_DIR}/olist_order_items_dataset.csv")
customers = pd.read_csv(f"{DATASET_DIR}/olist_customers_dataset.csv")
products  = pd.read_csv(f"{DATASET_DIR}/olist_products_dataset.csv")
sellers   = pd.read_csv(f"{DATASET_DIR}/olist_sellers_dataset.csv")
payments  = pd.read_csv(f"{DATASET_DIR}/olist_order_payments_dataset.csv")
reviews   = pd.read_csv(f"{DATASET_DIR}/olist_order_reviews_dataset.csv")
geo       = pd.read_csv(f"{DATASET_DIR}/olist_geolocation_dataset.csv")
cat_tr    = pd.read_csv(f"{DATASET_DIR}/product_category_name_translation.csv")

ts_cols = ["order_purchase_timestamp", "order_approved_at",
           "order_delivered_carrier_date", "order_delivered_customer_date",
           "order_estimated_delivery_date"]
for c in ts_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print("Đã nạp 9 bảng từ", DATASET_DIR)

## 1. Tổng quan dataset

In [ ]:
overview = {
    "orders": len(orders),
    "order_items": len(items),
    "customers (dòng)": len(customers),
    "customers (unique_id)": customers["customer_unique_id"].nunique(),
    "products": len(products),
    "sellers": len(sellers),
    "payments": len(payments),
    "reviews": len(reviews),
    "geolocation": len(geo),
}
print("Khoảng thời gian (purchase):",
      orders["order_purchase_timestamp"].min(), "->",
      orders["order_purchase_timestamp"].max())
pd.Series(overview, name="count").to_frame()

## 2. Profiling null / missing

Tỉ lệ null theo từng cột (chỉ liệt kê cột có null). Lưu ý: ngày giao null là hợp
lệ với đơn chưa giao; review comment null chiếm phần lớn (đa số khách không viết
nhận xét).

In [ ]:
def null_report(df, name):
    n = len(df)
    rep = df.isna().sum().to_frame("n_null")
    rep["pct_null"] = (rep["n_null"] / n * 100).round(2)
    rep = rep[rep["n_null"] > 0].sort_values("n_null", ascending=False)
    rep.insert(0, "table", name)
    return rep

null_df = pd.concat([null_report(df, nm) for df, nm in [
    (orders, "orders"), (items, "order_items"), (products, "products"),
    (reviews, "order_reviews"), (payments, "order_payments")] ])
null_df

In [ ]:
nd = orders["order_delivered_customer_date"].isna()
print("Đơn KHÔNG có ngày giao khách:", int(nd.sum()))
print("  -> trạng thái của các đơn đó:",
      orders.loc[nd, "order_status"].value_counts().to_dict())
print("Review thiếu comment message: %d (%.1f%%)" % (
    reviews["review_comment_message"].isna().sum(),
    reviews["review_comment_message"].isna().mean() * 100))

## 3. Phân phối các trường chính

In [ ]:
status_counts = orders["order_status"].value_counts()
plt.figure(figsize=(9, 5))
status_counts.plot(kind="bar", color="#4C72B0")
plt.title("Phân phối order_status"); plt.ylabel("số đơn"); plt.xticks(rotation=45)
for i, v in enumerate(status_counts):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
savefig("order_status.png")
status_counts.to_frame("count")

In [ ]:
plt.figure(figsize=(8, 5))
pt = payments["payment_type"].value_counts()
pt.plot(kind="bar", color="#55A868")
plt.title("Phân phối payment_type"); plt.ylabel("số lần thanh toán"); plt.xticks(rotation=30)
savefig("payment_type.png")
pt.to_frame("count")

In [ ]:
plt.figure(figsize=(7, 5))
rs = reviews["review_score"].value_counts().sort_index()
rs.plot(kind="bar", color="#C44E52")
plt.title("Phân phối review_score"); plt.xlabel("score"); plt.ylabel("số review")
savefig("review_score.png")
print("Điểm review trung bình:", round(reviews["review_score"].mean(), 2))
rs.to_frame("count")

In [ ]:
plt.figure(figsize=(11, 5))
om = orders.dropna(subset=["order_purchase_timestamp"]) \
           .set_index("order_purchase_timestamp").resample("ME").size()
om.plot(color="#4C72B0", marker="o")
plt.title("Số đơn theo tháng (order_purchase_timestamp)"); plt.ylabel("số đơn")
savefig("orders_by_month.png")

In [ ]:
deliv = orders.dropna(subset=["order_delivered_customer_date"]).copy()
deliv["delivery_days"] = (deliv["order_delivered_customer_date"]
                          - deliv["order_purchase_timestamp"]).dt.days
plt.figure(figsize=(10, 5))
deliv["delivery_days"].clip(lower=0, upper=60).plot(kind="hist", bins=60, color="#8172B3")
plt.title("Phân phối delivery_days (đơn đã giao, clip 0–60)"); plt.xlabel("ngày")
savefig("delivery_days.png")
print("delivery_days trung bình: %.2f ngày" % deliv["delivery_days"].mean())

## 4. Insight nghiệp vụ

In [ ]:
# Top category theo doanh thu (loại đơn canceled/unavailable, khớp mart_category_ranking)
valid = orders.loc[~orders["order_status"].isin(["canceled", "unavailable"]), ["order_id"]]
it = (items.merge(valid, on="order_id")
           .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
           .merge(cat_tr, on="product_category_name", how="left"))
it["cat"] = it["product_category_name_english"].fillna("unknown")
cat_rev = it.groupby("cat")["price"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
cat_rev[::-1].plot(kind="barh", color="#4C72B0")
plt.title("Top 10 category theo doanh thu (BRL, loại canceled/unavailable)"); plt.xlabel("doanh thu")
savefig("top_category_revenue.png")
cat_rev.round(2).to_frame("revenue")

In [ ]:
st = (orders.merge(customers[["customer_id", "customer_state"]], on="customer_id")
            ["customer_state"].value_counts().head(12))
plt.figure(figsize=(11, 5))
st.plot(kind="bar", color="#55A868")
plt.title("Top 12 bang theo số đơn"); plt.ylabel("số đơn")
savefig("orders_by_state.png")
st.to_frame("orders")

In [ ]:
# Late delivery rate (chỉ tính trên đơn ĐÃ GIAO, khớp mart_delivery_kpi = 8.11%)
delivered_mask = ((orders["order_status"] == "delivered")
                  | orders["order_delivered_customer_date"].notna())
delivered = orders[delivered_mask]
late = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
print("Đơn đã giao:      %d" % int(delivered_mask.sum()))
print("Đơn giao trễ:     %d" % int(late.sum()))
print("Late delivery rate: %.2f%%  (mẫu số = đơn đã giao)" % (late.sum() / delivered_mask.sum() * 100))

In [ ]:
# Repeat customer theo customer_unique_id
cu = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
per_cust = cu.groupby("customer_unique_id")["order_id"].nunique()
total_cust = per_cust.size
repeat = int((per_cust >= 2).sum())
print("Tổng khách (unique):     %d" % total_cust)
print("Khách mua lại (>=2 đơn): %d" % repeat)
print("Repeat rate:             %.2f%%" % (repeat / total_cust * 100))

## 5. Vấn đề chất lượng dữ liệu (DQ)

Các vấn đề thực tế gặp trong dữ liệu Olist — chính là cơ sở cho các quyết định
thiết kế DDL/PK ở tầng nguồn (`olist_source`).

In [ ]:
# (1) geolocation: 1 zip prefix -> nhiều toạ độ
print("geolocation: %d dòng | %d zip prefix phân biệt -> quan hệ 1-nhiều"
      % (len(geo), geo["geolocation_zip_code_prefix"].nunique()))
print("   => bảng nguồn cho surrogate PK (geolocation_id)\n")

# (2) review_id trùng nhưng (review_id, order_id) duy nhất
print("order_reviews: %d dòng | review_id phân biệt = %d | (review_id, order_id) phân biệt = %d"
      % (len(reviews), reviews["review_id"].nunique(),
         len(reviews.drop_duplicates(["review_id", "order_id"]))))
print("   => review_id KHÔNG unique, nhưng (review_id, order_id) unique => PK (review_id, order_id)\n")

# (3) anomaly: ngày giao < ngày mua
anom = int((orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print("orders có order_delivered_customer_date < order_purchase_timestamp (anomaly thời gian): %d" % anom)
print("   => simulator chỉ replay vòng đời khi delivery_month > purchase_month (guard anomaly)")

## 6. Bằng chứng pipeline (trích số đã kiểm chứng)

Các số dưới đây lấy từ kết quả chạy thật của pipeline lakehouse (không tính lại
từ CSV) — minh hoạ năng lực **incremental + time-travel + upsert**:

| Hạng mục | Kết quả đã verify |
|---|---|
| Step 0 — pipeline end-to-end | 10/10 task xanh, **pytest 17/17**, mọi DQ gate pass |
| Marts (full dataset) | late rate **8.11%**, doanh thu **13.49M BRL**, order_count **98,207** = 99,441 − 625 canceled − 609 unavailable |
| Phase 1 — incremental + time-travel | bronze.orders **4 → 328** qua 2 snapshot (`FOR VERSION AS OF`) |
| Phase 2 — CDC lifecycle + MERGE | Bronze **3024 dòng / 2580 distinct** (444 phiên bản trùng order_id) → Silver dedup **2580 unique** → Gold `MERGE INTO` **WHEN MATCHED** lật **444** đơn shipped→delivered tại chỗ; fact_orders **2580 không trùng**; rerun idempotent (count giữ nguyên) |

> Bronze giữ nhiều phiên bản (append CDC) còn Postgres luôn là current-state — đúng
> bản chất OLTP↔lakehouse. Silver `row_number() ... order by source_updated_at desc`
> chọn bản mới nhất; Gold MERGE upsert theo `order_id`.